# TreeRAG evals — MemWalker agent on a QuALITY-style multiple-choice test

Reads a CSV of human-written multiple-choice questions, has the prototype 7
navigation agent answer each one with **A / B / C / D** by walking the corpus
tree (no vector retrieval), scores accuracy, and ranks a big grid of traversal
variants on a **leaderboard you can sort by accuracy or by speed**.

**Question file** — `eval_questions.csv` (`.tsv`/`.xlsx` also work). One row per
question, columns: `id, question, A, B, C, D, answer`, where `answer` is the
correct letter. An optional `difficulty` column lets you slice easy vs hard
later, à la QuALITY's time-constrained split.

**What gets ablated** (each is a knob on the agent):

* **strategy** — `none` / `chunk` / `cluster` virtual-subfolder layouts;
* **working_memory** — keep useful notes seen en route (MemWalker) or not;
* **backtracking** — allow abandoning a dead-end branch or not;
* **thinking** — gpt-oss reasoning off vs on (usually the biggest latency lever);
* **group_summary** — describe virtual groups with cheap `heuristic` listings vs `llm` calls;
* **max_branch** — options per decision (fewer = more levels but easier picks);
* **decision_cap** — token cap on each short navigation decision;
* **nav_includes_options** *(new)* — also show the A–D choices during navigation;
* **breadcrumb** *(new)* — feed the path-of-summaries into the final answer;
* **vote_samples** *(new)* — self-consistency: sample the answer N times and majority-vote.

By default it runs a **one-axis-at-a-time** sweep (baseline + change one knob),
so each row reads as that factor's effect. `make_full_grid([...])` does a full
Cartesian over chosen axes when you want interactions. Every run is cached per
(config, question) so it's crash-resumable, and every individual result is kept
in `eval_cache/per_question.csv`; the leaderboard reports accuracy, mean/std time,
and mean/total input & output tokens.

Other ideas worth bolting on later: a small **beam width** instead of greedy
descent, per-child **relevance scoring** for very wide nodes, a **confidence
early-stop**, or a learned router. Each slots into `run_agent` / `_ask`.

In [1]:
%pip install ollama numpy pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, re, time, math, hashlib, textwrap
from pathlib import Path
from dataclasses import dataclass, field, asdict, replace
from typing import List, Dict, Any, Optional
from collections import Counter
from itertools import product
import ollama
import numpy as np
import pandas as pd

OLLAMA_URL  = "http://localhost:11528"
AGENT_MODEL = "gpt-oss:120b"          # makes the decisions and picks the final letter
EMBED_MODEL = "nomic-embed-text"      # only for the cluster strategy; groups similar files

CACHE_DIR  = Path("tree_cache"); TREE_FILE = CACHE_DIR / "corpus_tree.json"
QUESTIONS_FILE = "eval_questions.xlsx" # your xlsx of questions; reads every tab

MAX_EVIDENCE  = 12                    # leaves fed into the final answer prompt
MAX_QUESTIONS = None                  # set a small int to smoke test, then None for the real run
SCORE_MODE    = "judge"               # judge: agent writes a response, another llm grades it 0-1; or letter for A-D
KEEP_ALIVE    = "30m"
print(f"agent {AGENT_MODEL}; embed {EMBED_MODEL}; questions {QUESTIONS_FILE}")

agent gpt-oss:120b; embed nomic-embed-text; questions eval_questions.xlsx


In [3]:
client = ollama.Client(host=OLLAMA_URL, timeout=600)

def _model_names(r):
    raw = r.get("models", []) if hasattr(r, "get") else getattr(r, "models", [])
    out=[]
    for m in raw:
        n = getattr(m,"model",None) or getattr(m,"name",None)
        if n is None and isinstance(m,dict): n=m.get("model") or m.get("name")
        if n: out.append(n)
    return out

# waits for ollama at startup instead of crashing; agent model is required, embed only for cluster
def _wait_until_ready():
    announced=False
    while True:
        try:
            names=_model_names(client.list())
            if any(AGENT_MODEL in n for n in names):
                if not any(EMBED_MODEL in n for n in names):
                    print(f"note: {EMBED_MODEL} not present; the cluster strategy will fall back to chunk")
                print(f"ollama ok; {AGENT_MODEL} is loaded"); return
            reason=f"{AGENT_MODEL} not loaded yet"
        except Exception as e:
            reason=f"server unreachable; {type(e).__name__}: {e}"
        if not announced:
            print(f"waiting for ollama, {reason}; rechecking every 10s and wont stop"); announced=True
        time.sleep(10)

# one chat call to text; counts prompt and output tokens, and waits out any disconnect
def llm(prompt, cfg, counter, num_predict=None, temperature=0, think=None):
    use_think = cfg.thinking if think is None else think
    opts={"temperature":temperature, "num_predict": num_predict or cfg.decision_cap}
    attempt=0; pass_think=True
    while True:
        kw=dict(model=AGENT_MODEL, messages=[{"role":"user","content":prompt}],
                options=opts, keep_alive=KEEP_ALIVE)
        if pass_think: kw["think"]=use_think      # pass the bool either way; gpt-oss keeps reasoning unless told false
        try:
            r=client.chat(**kw)
            counter.calls+=1
            try: counter.in_tok  += int(r["prompt_eval_count"] or 0)
            except Exception: pass
            try: counter.out_tok += int(r["eval_count"] or 0)
            except Exception: pass
            txt=(r["message"]["content"] or "").strip()
            if not txt:                           # reasoning models sometimes leave content empty; use the thinking
                try: txt=(r["message"]["thinking"] or "").strip()
                except Exception: pass
            return txt
        except TypeError:
            pass_think=False                      # client too old for the think arg; drop it
        except Exception as e:
            attempt+=1
            if attempt==1 or attempt%5==0: print(f"[waiting for ollama] {type(e).__name__}: {e}; retrying")
            time.sleep(min(60, 5*2**min(attempt-1,4)))

# embeddings for clustering only; never used to rank answers
def embed(text, counter):
    try:
        r=client.embeddings(model=EMBED_MODEL, prompt=text or " ")
        try: counter.in_tok += int(r.get("prompt_eval_count",0) or 0)
        except Exception: pass
        return r["embedding"]
    except Exception:
        return None

_wait_until_ready()
print("llm and embed helpers ready")

ollama ok; gpt-oss:120b is loaded
llm and embed helpers ready


In [4]:
import re, math, hashlib
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional
from collections import Counter
import numpy as np


@dataclass
class TreeNode:
    node_id:str; node_type:str; name:str; path:str; summary:str
    content:str=""; children:List["TreeNode"]=field(default_factory=list)
    metadata:Dict[str,Any]=field(default_factory=dict)

    @classmethod
    def from_dict(cls,d):
        n=cls(node_id=d["node_id"],node_type=d["node_type"],name=d["name"],
              path=d.get("path",""),summary=d.get("summary",""),
              content=d.get("content",""),metadata=d.get("metadata",{}))
        n.children=[cls.from_dict(c) for c in d.get("children",[])]
        return n

    def is_leaf(self): return self.node_type=="chunk"
    def count_leaves(self): return 1 if self.is_leaf() else sum(c.count_leaves() for c in self.children)


@dataclass
class Question:
    qid:str; stem:str; options:Dict[str,str]; answer:str; difficulty:str=""


@dataclass(frozen=True)
class Config:
    strategy:str="cluster"          # none chunk or cluster; how wide nodes get grouped
    working_memory:bool=True        # keep useful notes seen en route, memwalker style
    backtracking:bool=True          # allow going back up a dead end branch
    thinking:bool=False             # gpt-oss reasoning; usually the big latency lever
    group_summary:str="heuristic"   # describe virtual groups cheaply or with an llm call
    max_branch:int=6                # options per decision; fewer is easier but deeper
    decision_cap:int=512            # token cap on each nav decision; they are short
    answer_cap:int=8                # token cap on the final letter
    nav_includes_options:bool=False # new idea, show the abcd choices during navigation too
    breadcrumb:bool=False           # new idea, feed the path summaries into the final answer
    vote_samples:int=1              # new idea, self consistency; sample the answer n times and vote
    max_steps:int=24                # safety budget per question


# what makes this config differ from the plain default; used as a readable label
def config_name(cfg):
    base=Config()
    diffs=[f"{k}={getattr(cfg,k)}" for k in cfg.__dataclass_fields__ if getattr(cfg,k)!=getattr(base,k)]
    return "baseline" if not diffs else ", ".join(diffs)

def config_key(cfg):
    return hashlib.md5(repr(asdict(cfg)).encode()).hexdigest()[:12]


class Counters:
    def __init__(self): self.in_tok=0; self.out_tok=0; self.calls=0


def clip(t,n):
    t=re.sub(r"\s+"," ",t or "").strip()
    return t if len(t)<=n else t[:n]+" …"


# ---- virtual subfolders; only place embeddings are touched and only to organise groups ----
_nav_cache={}; _embed_cache={}; _gsum_cache={}

def reset_vcaches():
    _nav_cache.clear(); _gsum_cache.clear()   # embeddings persist; they dont depend on the config

def _embed_node(node,counter):
    if node.node_id in _embed_cache: return _embed_cache[node.node_id]
    v=embed((node.name+". "+(node.summary or ""))[:2000],counter)
    _embed_cache[node.node_id]=v; return v

# tiny kmeans on unit vectors so its cosine ish
def _kmeans(vectors,k,iters=25,seed=0):
    X=np.asarray(vectors,dtype=float); n=len(X); k=max(1,min(k,n))
    X=X/np.clip(np.linalg.norm(X,axis=1,keepdims=True),1e-9,None)
    rng=np.random.default_rng(seed); C=X[rng.choice(n,size=k,replace=False)].copy()
    labels=np.full(n,-1)
    for _ in range(iters):
        d=((X[:,None,:]-C[None,:,:])**2).sum(-1); new=d.argmin(1)
        if np.array_equal(new,labels): break
        labels=new
        for j in range(k):
            pts=X[labels==j]; C[j]=pts.mean(0) if len(pts) else X[rng.integers(n)]
    return labels.tolist()

def _group_summary(members,label,cfg,counter):
    key=(label,cfg.group_summary,tuple(m.node_id for m in members))
    if key in _gsum_cache: return _gsum_cache[key]
    if cfg.group_summary=="llm":
        joined="\n".join(f"- {m.summary}" for m in members)
        prompt=("in 2-4 sentences say what this group called '"+label+"' covers so an agent can "
                "decide whether to explore it; list the main topics. respond with only the text.\n\n"+joined)
        s=llm(prompt,cfg,counter,num_predict=300)
    else:
        lines=[f"- {m.name}: {clip(m.summary,160)}" for m in members[:cfg.max_branch]]
        more=f"\n- …and {len(members)-cfg.max_branch} more" if len(members)>cfg.max_branch else ""
        s=f"a group of {len(members)} related items:\n"+"\n".join(lines)+more
    _gsum_cache[key]=s; return s

def _vid(pid,tag): return "v"+hashlib.md5(f"{pid}|{tag}".encode()).hexdigest()[:11]

def _make_vgroup(parent,members,idx,cfg,counter):
    return TreeNode(node_id=_vid(parent.node_id,f"{cfg.strategy}:{cfg.max_branch}:{idx}"),
                    node_type="vgroup",name=f"[group {idx+1} · {len(members)} items]",path="",
                    summary=_group_summary(members,f"{parent.name} group {idx+1}",cfg,counter),
                    children=list(members),metadata={"virtual":True})

def _chunk_groups(parent,kids,cfg,counter):
    size=math.ceil(len(kids)/cfg.max_branch)              # at most max_branch groups
    groups=[kids[i:i+size] for i in range(0,len(kids),size)]
    return [_make_vgroup(parent,g,i,cfg,counter) for i,g in enumerate(groups)]

def _cluster_groups(parent,kids,cfg,counter):
    vecs=[_embed_node(k,counter) for k in kids]
    if any(v is None for v in vecs): return _chunk_groups(parent,kids,cfg,counter)  # no embeds; fall back
    k=max(2,min(cfg.max_branch,math.ceil(len(kids)/max(2,cfg.max_branch-2))))
    labels=_kmeans(vecs,k); buckets={}
    for kid,lab in zip(kids,labels): buckets.setdefault(lab,[]).append(kid)
    if len(buckets)<2 or max(len(v) for v in buckets.values())==len(kids):
        return _chunk_groups(parent,kids,cfg,counter)     # degenerate; chunk it so we make progress
    ordered=[buckets[lab] for lab in sorted(buckets)]
    return [_make_vgroup(parent,m,i,cfg,counter) for i,m in enumerate(ordered)]

# the children the agent chooses among, virtualised down to max_branch
def get_nav_children(node,cfg,counter):
    key=(node.node_id,cfg.strategy,cfg.max_branch,cfg.group_summary)
    if key in _nav_cache: return _nav_cache[key]
    kids=node.children
    if cfg.strategy=="none" or len(kids)<=cfg.max_branch: res=list(kids)
    elif cfg.strategy=="chunk": res=_chunk_groups(node,kids,cfg,counter)
    elif cfg.strategy=="cluster": res=_cluster_groups(node,kids,cfg,counter)
    else: res=list(kids)
    _nav_cache[key]=res; return res


# ---- the agent ----
import json as _json

def _add_memory(mem,fact,cap=20):
    fact=clip(fact,500)
    if fact and fact.lower() not in ("","none","n/a") and fact not in mem:
        mem.append(fact); del mem[:-cap]

# pull a single action out of the model json, tolerant of junk
def _parse_decision(raw,n_opts,wm):
    s=re.sub(r"^```(?:json)?|```$","",raw.strip(),flags=re.M).strip()
    m=re.search(r"\{.*\}",s,flags=re.S)
    if m:
        try:
            d=_json.loads(m.group(0)); act=str(d.get("action","")).lower().strip()
            if act not in ("descend","answer","backtrack"): act="descend" if n_opts else "answer"
            ci=d.get("child",None)
            try: ci=int(ci)
            except (TypeError,ValueError): ci=None
            return {"action":act,"child":ci,"remember":(d.get("remember") or "").strip() if wm else ""}
        except Exception: pass
    return {"action":"descend" if n_opts else "backtrack","child":0 if n_opts else None,"remember":""}

def _ask(query,node,options,memory,cfg,counter,can_back):
    opts_txt="\n".join(f"[{i}] {o.name} — {clip(o.summary,360)}" for i,o in enumerate(options)) or "(no options here)"
    acts=(['"descend"'] if options else [])+['"answer"']+(['"backtrack"'] if can_back else [])
    mem_block=""; remember_field=""
    if cfg.working_memory:
        mem_block="WORKING MEMORY (facts gathered so far):\n"+("\n".join("- "+m for m in memory) or "(empty)")+"\n\n"
        remember_field='"remember": "<a useful fact from the CURRENT summary to keep, else empty>", '
    prompt=(
        "you are an agent navigating a tree of document summaries to answer a question. "
        "you see only summaries and move one node at a time.\n\n"
        f"QUESTION: {query}\n\n{mem_block}"
        f"CURRENT NODE: {node.name} [{node.node_type}]\nCURRENT SUMMARY: {clip(node.summary,900)}\n\n"
        f"CHILD OPTIONS:\n{opts_txt}\n\n"
        f"choose ONE action ({', '.join(acts)}). reply with ONLY json:\n"
        '{"reasoning":"<1 sentence>", '+remember_field+
        '"action":"descend|answer|backtrack", "child":<index or null>}\n'
        "descend into the option most likely to lead to the answer; answer if you have enough; "
        "backtrack if none of these are relevant.")
    cap=cfg.decision_cap if not cfg.thinking else max(cfg.decision_cap,1024)
    return _parse_decision(llm(prompt,cfg,counter,num_predict=cap),len(options),cfg.working_memory)

SCORE_MODE = globals().get("SCORE_MODE","judge")   # judge: free response graded 0-1 by an llm; letter: pick A-D

# a short label for the path printout
def _short(n):
    if n.metadata.get("virtual"): return "[grp]"
    return (n.name or "?")[:22]

# run the whole traversal for one question; returns the response plus a trace of where it went
def run_agent(q,cfg,counter,trace=True):
    nav_q=q.stem
    if cfg.nav_includes_options:
        nav_q+="\noptions: "+"; ".join(f"{L}) {q.options[L]}" for L in "ABCD")
    memory=[]; evidence=[]; seen=set(); visited={ROOT.node_id}; crumbs=[]
    trail=["root"]; backtracks=0
    stack=[{"node":ROOT,"options":get_nav_children(ROOT,cfg,counter),"tried":set()}]
    steps=0
    while stack and steps<cfg.max_steps:
        steps+=1; fr=stack[-1]; node=fr["node"]
        present=[(i,o) for i,o in enumerate(fr["options"]) if i not in fr["tried"] and o.node_id not in visited]
        opts=[o for _,o in present]
        can_back=cfg.backtracking and len(stack)>1
        dec=_ask(nav_q,node,opts,memory,cfg,counter,can_back)
        if cfg.working_memory and dec["remember"]: _add_memory(memory,dec["remember"])
        if cfg.breadcrumb: crumbs.append(node.summary)
        act=dec["action"]
        if act=="answer" and not evidence and opts: act="descend"   # dont answer before reading a real document
        if act=="descend" and not opts:                             # nothing left here; go back up or stop
            act="backtrack" if can_back else "answer"
        if act=="answer" or (act=="backtrack" and not can_back):    # disallowed backtrack becomes answer
            if node.node_id not in seen: evidence.append(node); seen.add(node.node_id)
            break
        if act=="backtrack":
            popped=stack.pop(); parent=stack[-1]; backtracks+=1; trail.append("↩")
            for i,o in enumerate(parent["options"]):
                if o.node_id==popped["node"].node_id: parent["tried"].add(i); break
            continue
        ci=dec["child"]
        if ci is None or not (0<=ci<len(opts)): ci=0    # bad index; just take the first
        child=opts[ci]; visited.add(child.node_id)
        child_opts=get_nav_children(child,cfg,counter)
        if child.is_leaf() or not child_opts:           # reached a real leaf; grab its text
            if child.node_id not in seen: evidence.append(child); seen.add(child.node_id)
            if cfg.working_memory and child.content: _add_memory(memory,clip(child.content,600))
            trail.append(_short(child)+"*")
            for i,o in enumerate(fr["options"]):
                if o.node_id==child.node_id: fr["tried"].add(i); break
            continue
        trail.append(_short(child))
        stack.append({"node":child,"options":child_opts,"tried":set()})
    if SCORE_MODE=="letter":
        response=_answer_letter(q,memory,evidence,crumbs,cfg,counter)
    else:
        response=_answer_response(q,memory,evidence,crumbs,cfg,counter)
    return {"response":response,"evidence":evidence,"steps":steps,
            "backtracks":backtracks,"path":" › ".join(trail)}

def _parse_letter(text):
    t=(text or "").strip().upper()
    if not t: return ""
    cues=re.findall(r"ANSWER[^ABCD]{0,8}?\b([ABCD])\b",t)   # answer is B, correct answer: C; take the last
    if cues: return cues[-1]
    m=re.search(r"\b([ABCD])\b",t) or re.search(r"([ABCD])",t)
    return m.group(1) if m else ""

# build the final multiple choice prompt from whatever was gathered, then vote if asked
def _answer_letter(q,memory,evidence,crumbs,cfg,counter):
    opts="\n".join(f"{L}. {q.options[L]}" for L in "ABCD")
    parts=[]
    if cfg.working_memory and memory: parts.append("notes you gathered:\n"+"\n".join("- "+m for m in memory))
    if cfg.breadcrumb and crumbs: parts.append("summaries along your path:\n"+"\n".join("- "+clip(c,200) for c in crumbs[-8:]))
    if evidence:
        ev="\n\n".join(f"[{e.metadata.get('source_file') or e.path or e.name}] {clip(e.content or e.summary,1200)}"
                       for e in evidence[:MAX_EVIDENCE])
        parts.append("evidence:\n"+ev)
    ctx="\n\n".join(parts) or "(no context gathered)"
    prompt=(f"answer the multiple choice question using the gathered information.\n\n"
            f"QUESTION: {q.stem}\nOPTIONS:\n{opts}\n\n{ctx}\n\n"
            "respond with ONLY the single letter of the best option: A, B, C, or D.")
    cap=1024 if cfg.thinking else 64
    if cfg.vote_samples<=1:
        return _parse_letter(llm(prompt,cfg,counter,num_predict=cap,temperature=0))
    votes=[_parse_letter(llm(prompt,cfg,counter,num_predict=cap,temperature=0.7)) for _ in range(cfg.vote_samples)]
    votes=[v for v in votes if v]
    return Counter(votes).most_common(1)[0][0] if votes else ""

# free text answer from whatever the traversal gathered; this is what the judge grades
def _answer_response(q,memory,evidence,crumbs,cfg,counter):
    parts=[]
    if cfg.working_memory and memory: parts.append("notes you gathered:\n"+"\n".join("- "+m for m in memory))
    if cfg.breadcrumb and crumbs: parts.append("summaries along your path:\n"+"\n".join("- "+clip(c,200) for c in crumbs[-8:]))
    if evidence:
        ev="\n\n".join(f"[{e.metadata.get('source_file') or e.path or e.name}] {clip(e.content or e.summary,1200)}"
                       for e in evidence[:MAX_EVIDENCE])
        parts.append("evidence:\n"+ev)
    ctx="\n\n".join(parts) or "(no context gathered)"
    prompt=(f"answer the question using only the gathered information; be specific.\n\n"
            f"QUESTION: {q.stem}\n\n{ctx}\n\n"
            "give the answer in 1-3 sentences, and cite the source file in brackets if relevant.")
    return llm(prompt,cfg,counter,num_predict=1024 if cfg.thinking else 320,temperature=0)

_JUDGE_CFG=Config()     # judge runs with reasoning off and short output

# grade the agents free response against the gold option, 0 to 1
def judge_score(q,response,counter):
    gold=q.options.get(q.answer,"")
    opts="\n".join(f"{L}. {q.options[L]}" for L in "ABCD")
    prompt=(f"you are grading an answer to a multiple choice question.\n\n"
            f"QUESTION: {q.stem}\nOPTIONS:\n{opts}\nCORRECT ANSWER: {q.answer}. {gold}\n\n"
            f"STUDENT RESPONSE:\n{clip(response,1200)}\n\n"
            "rate from 0.0 to 1.0 how correct the student response is; 1.0 means it clearly conveys the "
            "correct answer, 0.0 means wrong or irrelevant, and partial credit is fine. respond with ONLY the number.")
    raw=llm(prompt,_JUDGE_CFG,counter,num_predict=16,temperature=0)
    m=re.search(r"(0?\.\d+|0|1(?:\.0+)?)", raw or "")
    return max(0.0,min(1.0,float(m.group(1)))) if m else 0.0

print("eval core ready; configs, virtual subfolders and the agent are defined")

eval core ready; configs, virtual subfolders and the agent are defined


In [5]:
if not TREE_FILE.exists():
    raise SystemExit(f"tree not found at {TREE_FILE.resolve()}; build it first with prototype 5")
ROOT = TreeNode.from_dict(json.loads(TREE_FILE.read_text(encoding="utf-8")))
print(f"loaded tree {ROOT.name}; {ROOT.count_leaves()} leaves and {len(ROOT.children)} top level children")

loaded tree folders; 95458 leaves and 10 top level children


In [6]:
import json, time, re
from pathlib import Path
import pandas as pd
try:
    from tqdm.auto import tqdm as _tqdm           # nice bar in jupyter, with eta
except Exception:
    _tqdm=None                                    # plain fallback if tqdm isnt installed

def _blank(v):
    if v is None: return True
    s=str(v).strip(); return s=="" or s.lower()=="nan"

# pull a b c d out of text that may hold one option or all four, one per line
def _parse_options_cell(text):
    opts={}
    for line in re.split(r"[\r\n;]+", str(text)):
        mm=re.match(r"^\s*\(?\s*([A-Da-d])\s*[).:\-\u2013]\s*(.+)$", line.strip())
        if mm and mm.group(1).upper() not in opts: opts[mm.group(1).upper()]=mm.group(2).strip()
    return opts

def _match(df):
    norm={re.sub(r"[^a-z]","",str(c).lower()):c for c in df.columns}
    def col(*names):
        for n in names:
            if n in norm: return norm[n]
        return None
    return dict(
        q=col("question","q","prompt","stem"),
        ans=col("correctanswer","answer","correct","gold","label","key"),
        idc=col("number","id","qid","index"),
        diff=col("difficulty","level"),
        combined=col("mcoptions","options","choices","mcq","multiplechoiceoptions","mcoptionsabcd"),
        A=col("a","optiona","choicea"), B=col("b","optionb","choiceb"),
        C=col("c","optionc","choicec"), D=col("d","optiond","choiced"))

# spreadsheets often put a title row above the real header; find the header by keywords
def _find_header(raw):
    best,score=0,-1
    for r in range(min(8,len(raw))):
        cells=[re.sub(r"[^a-z]","",str(x).lower()) for x in raw.iloc[r].tolist()]
        s=sum(any(k in c for k in ("question","answer","option","number","choice")) for c in cells)
        if s>score: score,best=s,r
    return best

def _prep(raw):
    hr=_find_header(raw)
    df=raw.iloc[hr+1:].copy(); df.columns=[str(c) for c in raw.iloc[hr].tolist()]
    return df.reset_index(drop=True)

# turn one sheet into questions; handles split columns, one cell of choices, or choices merged across rows
def _rows_from_df(df, sheet, out, seen):
    m=_match(df)
    has_split=all(m[k] for k in ("A","B","C","D"))
    if not (m["q"] and m["ans"] and (has_split or m["combined"])):
        print(f"  skip sheet '{sheet}'; columns found were {list(df.columns)}"); return

    def add(qid,stem,opts,ansv,diffv):
        if _blank(stem) or any(L not in opts or _blank(opts[L]) for L in "ABCD") or _blank(ansv):
            return False
        am=re.search(r"[A-Da-d]",str(ansv))
        if not am: return False
        qid=str(qid).strip() if not _blank(qid) else f"{sheet}_{len(out)+1}"
        while qid in seen: qid+="_x"                       # keep ids unique across tabs
        seen.add(qid)
        out.append(Question(qid,str(stem).strip(),{L:opts[L] for L in "ABCD"},
                            am.group(0).upper(),"" if _blank(diffv) else str(diffv).strip()))
        return True

    kept=0
    if has_split:
        for _,r in df.iterrows():
            opts={L:str(r[m[L]]).strip() for L in "ABCD"}
            kept+=add(r[m["idc"]] if m["idc"] else None, r[m["q"]], opts, r[m["ans"]],
                      r[m["diff"]] if m["diff"] else None)
    else:
        keycol=m["idc"] or m["q"]                          # a new question starts where this is non blank
        group=df[keycol].apply(lambda x: not _blank(x)).cumsum()
        for _,sub in df.groupby(group):
            def first(c):                                  # first non blank in the merged block
                for v in sub[c].tolist():
                    if not _blank(v): return v
                return None
            lines="\n".join(str(v) for v in sub[m["combined"]].tolist() if not _blank(v))
            opts=_parse_options_cell(lines)
            kept+=add(first(m["idc"]) if m["idc"] else None, first(m["q"]), opts, first(m["ans"]),
                      first(m["diff"]) if m["diff"] else None)
    print(f"  sheet '{sheet}': kept {kept}")

# read the question file; reads every tab of an xlsx and matches columns loosely
def load_questions(path):
    p=Path(path); ext=p.suffix.lower(); out=[]; seen=set()
    if ext in (".xlsx",".xls"):
        for name,raw in pd.read_excel(p, sheet_name=None, header=None).items():   # every tab, raw
            _rows_from_df(_prep(raw), name, out, seen)
    elif ext==".tsv":
        _rows_from_df(_prep(pd.read_csv(p,sep="\t",header=None,dtype=object)), "tsv", out, seen)
    else:
        _rows_from_df(_prep(pd.read_csv(p,header=None,dtype=object)), "csv", out, seen)
    if not out:
        raise ValueError("no valid questions parsed; need question, options and a correct answer letter")
    print(f"parsed {len(out)} questions total from {p.name}")
    return out


EVAL_CACHE_DIR=Path("eval_cache"); RESULTS_FILE=EVAL_CACHE_DIR/"results.json"
CACHE={}

def load_cache():
    global CACHE
    if RESULTS_FILE.exists():
        CACHE=json.loads(RESULTS_FILE.read_text())
    print(f"results cache: {len(CACHE)} stored runs")

def save_cache():
    EVAL_CACHE_DIR.mkdir(exist_ok=True)
    tmp=RESULTS_FILE.with_suffix(".tmp")          # write then swap so a crash mid write cant corrupt it
    tmp.write_text(json.dumps(CACHE)); tmp.replace(RESULTS_FILE)

class _PlainBar:
    def __init__(self,total,initial,desc): self.t=total; self.n=initial; self.d=desc
    def update(self,k=1): self.n+=k; print(f"\r{self.d} {self.n}/{self.t}",end="",flush=True)
    def set_postfix(self,**k): pass
    def set_description(self,d): self.d=d
    def close(self): print()

def _bar(total,initial,desc):
    if _tqdm is not None:
        return _tqdm(total=total,initial=initial,unit="q",desc=desc,dynamic_ncols=True)
    return _PlainBar(total,initial,desc)

import random as _random

# answer one question under one config, then score it; judge mode grades a free response 0-1
def _eval_one(cfg,q):
    c=Counters(); t0=time.perf_counter()
    res=run_agent(q,cfg,c,trace=True); dt=time.perf_counter()-t0
    if SCORE_MODE=="letter":
        score=float(res["response"]==q.answer)
    else:
        score=judge_score(q,res["response"],Counters())          # judge tokens kept out of the config totals
    srcs=[]
    for e in res["evidence"]:
        s=e.metadata.get("source_file") or e.path or e.name
        if s and s not in srcs: srcs.append(s)
    return {"config":config_name(cfg),"key":config_key(cfg),"qid":q.qid,
            "response":res["response"],"gold":q.answer,"score":round(score,3),
            "correct":int(score>=0.5),"time":round(dt,3),
            "in_tok":c.in_tok,"out_tok":c.out_tok,"calls":c.calls,
            "steps":res["steps"],"backtracks":res["backtracks"],
            "path":res["path"],"sources":", ".join(srcs[:3]),"difficulty":q.difficulty}

# one config over the questions; cached per config and question so its crash resumable
def run_config(cfg,questions,max_q=None,verbose=False):
    reset_vcaches(); qs=questions[:max_q] if max_q else questions; rows=[]; key=config_key(cfg)
    for q in qs:
        ck=f"{key}::{q.qid}"
        if ck in CACHE: rec=CACHE[ck]
        else: rec=_eval_one(cfg,q); CACHE[ck]=rec; save_cache()
        rows.append(rec)
    return rows

# print a readable block per question so you can see the path, the file found, and why it was right or wrong
def _print_diag(q,rec):
    o=q.options
    print(f"\n{q.qid}  gold={rec['gold']}  judge_score={rec['score']:.2f}")
    print(f"  choices: A) {clip(o['A'],52)} | B) {clip(o['B'],52)} | C) {clip(o['C'],52)} | D) {clip(o['D'],52)}")
    print(f"  correct: {rec['gold']}) {clip(o[rec['gold']],80)}")
    print(f"  path: {rec['path']}   ({rec['steps']} steps, {rec['backtracks']} backtracks)")
    print(f"  found: {rec['sources'] or '— nothing reached —'}")
    print(f"  agent: {clip(rec['response'],180)}")

# run n random questions under one config, with the full traversal traced and judge scored
def run_sample(questions, n=25, cfg=None, seed=0, verbose=True):
    cfg=cfg or Config(); reset_vcaches()
    sample=_random.Random(seed).sample(questions, min(n,len(questions)))
    print(f"sampling {len(sample)} questions under: {config_name(cfg)}\n"+"="*70)
    rows=[]; tot=0.0
    bar=_bar(len(sample),0,"sample")
    for q in sample:
        rec=_eval_one(cfg,q); rows.append(rec); tot+=rec["score"]; bar.update(1)
        if verbose: _print_diag(q,rec)
    bar.close()
    print("\n"+"="*70+f"\nmean judge score over {len(sample)} questions: {tot/len(sample):.3f}")
    return rows

# run the whole grid with a progress bar and eta; rerun after a crash and it resumes from the cache
def run_grid(grid,questions,max_q=None,verbose=False):
    qs=questions[:max_q] if max_q else questions
    tasks=[(cfg,q) for cfg in grid for q in qs]
    done=sum(1 for cfg,q in tasks if f"{config_key(cfg)}::{q.qid}" in CACHE)   # already in the cache
    if done: print(f"resuming; {done}/{len(tasks)} runs already cached, the bar starts from there")
    bar=_bar(len(tasks),done,"eval")
    all_rows=[]; score_sum=0.0; n=0; last=None
    for cfg,q in tasks:
        if cfg is not last:
            reset_vcaches(); last=cfg                  # fresh group summaries so tokens stay per config
            bar.set_description((config_name(cfg)[:34]))
        ck=f"{config_key(cfg)}::{q.qid}"
        if ck in CACHE:
            rec=CACHE[ck]                              # resumed; already counted in the bars start
        else:
            rec=_eval_one(cfg,q); CACHE[ck]=rec; save_cache(); bar.update(1)
        all_rows.append(rec); score_sum+=rec["score"]; n+=1
        bar.set_postfix(score=f"{score_sum/n:.2f}",last=f"{rec['score']}")
    bar.close()
    print(f"done; {n} results, overall mean score {score_sum/n:.3f} across all configs")
    return all_rows


# collapse the per question rows into one line per config with means and spreads
def leaderboard(all_rows):
    df=pd.DataFrame(all_rows)
    g=df.groupby(["config","key"],sort=False)
    lb=g.agg(n=("score","size"),correct=("correct","sum"),
             accuracy=("score","mean"),
             mean_time=("time","mean"),std_time=("time","std"),
             mean_in=("in_tok","mean"),std_in=("in_tok","std"),
             mean_out=("out_tok","mean"),std_out=("out_tok","std"),
             total_in=("in_tok","sum"),total_out=("out_tok","sum"),
             mean_calls=("calls","mean")).reset_index()
    lb["accuracy_pct"]=(lb["accuracy"]*100).round(1)
    for c in ["mean_time","std_time","mean_in","std_in","mean_out","std_out","mean_calls"]:
        lb[c]=lb[c].fillna(0).round(2)
    return lb

def by_accuracy(lb):
    return lb.sort_values(["accuracy","mean_time"],ascending=[False,True]).reset_index(drop=True)

def by_speed(lb):
    return lb.sort_values(["mean_time","accuracy"],ascending=[True,False]).reset_index(drop=True)

# write the leaderboard and every individual run so nothing is lost
def save_outputs(lb,all_rows,outdir="eval_cache"):
    d=Path(outdir); d.mkdir(exist_ok=True)
    cols=["config","accuracy_pct","correct","n","mean_time","std_time",
          "mean_in","mean_out","total_in","total_out","mean_calls","key"]
    by_accuracy(lb)[cols].to_csv(d/"leaderboard_by_accuracy.csv",index=False)
    by_speed(lb)[cols].to_csv(d/"leaderboard_by_speed.csv",index=False)
    pd.DataFrame(all_rows).to_csv(d/"per_question.csv",index=False)
    print(f"wrote {d/'leaderboard_by_accuracy.csv'}, leaderboard_by_speed.csv and per_question.csv")

print("eval harness ready; load_questions, run_grid, leaderboard, by_accuracy, by_speed")

/opt/homebrew/Cellar/jupyterlab/4.5.7_1/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


eval harness ready; load_questions, run_grid, leaderboard, by_accuracy, by_speed


In [7]:
from dataclasses import replace
from itertools import product

# every axis we ablate and the alternatives we try for each
AXES={
    "strategy":["none","chunk","cluster"],
    "working_memory":[False,True],
    "backtracking":[False,True],
    "thinking":[False,True],
    "group_summary":["heuristic","llm"],
    "max_branch":[4,6,8],
    "decision_cap":[256,512],
    "nav_includes_options":[False,True],
    "breadcrumb":[False,True],
    "vote_samples":[1,3],
}

# baseline plus, for each axis, a config that changes only that one thing; clean to read off effects
def make_oat_grid(base=None,axes=None):
    base=base or Config(); axes=axes or AXES
    seen={config_key(base)}; grid=[base]
    for axis,vals in axes.items():
        for v in vals:
            cfg=replace(base,**{axis:v}); k=config_key(cfg)
            if k not in seen: seen.add(k); grid.append(cfg)
    return grid

# full cartesian over only the axes you name; warn yourself, this explodes fast
def make_full_grid(axes_subset):
    keys=list(axes_subset); base=Config(); seen=set(); grid=[]
    for combo in product(*[AXES[k] for k in keys]):
        cfg=replace(base,**dict(zip(keys,combo))); k=config_key(cfg)
        if k not in seen: seen.add(k); grid.append(cfg)
    return grid

print("grid builders ready; make_oat_grid for the standard sweep, make_full_grid for cartesian")

grid builders ready; make_oat_grid for the standard sweep, make_full_grid for cartesian


In [8]:
load_cache()
questions = load_questions(QUESTIONS_FILE)
print(f"loaded {len(questions)} questions")

# diagnostic: 25 random questions under the full TreeRAG agent, traced and judge scored
# Config() defaults are the full approach: cluster subfolders, working memory on, backtracking on
sample_rows = run_sample(questions, n=25, cfg=Config(), seed=0)

results cache: 0 stored runs
  sheet 'General QMS': kept 47
  sheet 'Genomics': kept 115
  sheet 'Tissue Portal': kept 53
  sheet 'GSI': kept 0
parsed 215 questions total from eval_questions.xlsx
loaded 215 questions
sampling 25 questions under: baseline


sample:   4%|████                                                                                                | 1/25 [01:52<45:09, 112.91s/q]


GEN-53  gold=B  judge_score=0.00
  choices: A) When we have experienced this same type of NC before … | B) When the SOP you are following contains instructions … | C) When only RUO samples were affected. | D) When only clinical samples were affected.
  correct: B) When the SOP you are following contains instructions on how to handle that speci …
  path: root › [grp] › ↩ › [grp] › Quality SOPs and Works › [grp] › [grp] › Temperature Quality Co › Temperature Quality Co › Temperature Quality Co* › ↩ › Temperature Quality Co › [grp] › Temperature Quality Co › ↩ › Temperature Quality Co › [grp] › [grp] › [grp] › Temperature Quality Co* › Temperature Quality Co* › ↩ › [grp] › Temperature Quality Co* › ↩   (24 steps, 5 backtracks)
  found: folders/Quality SOPs and Worksheets/Temperature Quality Control Procedure.docx
  agent: You do not need to report a non‑conformance when the laboratory temperature remains within the specified range (19 °C – 25 °C) and no deviation is observed; only detect

sample:   8%|████████                                                                                             | 2/25 [02:48<30:17, 79.02s/q]


TP-37  gold=B  judge_score=0.00
  choices: A) Four deep well plates and one regular 200uL plate | B) Five deep well plates and one regular 200uL plate | C) Four deep well plates, one 200uL elution plate and o … | D) Three deep well plates and two regular 200uL plates
  correct: B) Five deep well plates and one regular 200uL plate
  path: root › [grp] › Technical SOPs and Wor › [grp] › [grp] › DNA Extraction from Bu › DNA Extraction from Bu › DNA Extraction from Bu* › DNA Extraction from Bu* › ↩ › DNA Extraction from Bu › DNA Extraction from Bu* › ↩ › DNA Extraction from Bu › DNA Extraction from Bu › DNA Extraction from Bu*   (16 steps, 2 backtracks)
  found: folders/Technical SOPs and Worksheets/DNA Extraction from Buffy Coat - KingFisher Method.docx
  agent: You need four deep‑well plates – a Wash I Solution plate, Wash II Solution plate 1, Wash II Solution plate 2 and an Elution plate – and a 96‑tip comb (placed in a standard plate) f …


sample:  12%|████████████                                                                                         | 3/25 [04:20<31:08, 84.91s/q]


GEN-62  gold=A  judge_score=0.00
  choices: A) We don't have one - we just use OICR's org chart. | B) Our chart must be updated prior to every audit | C) The chart can be found in QM-017 Laboratory Scope, R … | D) The purpose of the chart is to show relationships be …
  correct: A) We don't have one - we just use OICR's org chart.
  path: root › [grp] › Management › [grp] › [grp] › Accreditation › ↩ › ↩ › ↩ › [grp] › ↩ › [grp] › ↩ › [grp] › ↩ › ↩ › CAPA › ↩ › ↩ › [grp] › ↩ › [grp] › ↩   (23 steps, 11 backtracks)
  found: folders
  agent: The statement that the OICR Genomics organizational chart is **not** stored in the Management folder is false – the chart is expected to be one of the governance documents kept wit …


KeyboardInterrupt: 

In [ ]:
# the full ablation sweep; judge scored, progress bar with eta, resumes from cache after a crash
GRID = make_oat_grid()                 # baseline plus one change per axis; or make_full_grid(["strategy","thinking"])
print(f"{len(GRID)} configs to run; ~{len(GRID)*len(questions)} agent runs before caching")

all_rows = run_grid(GRID, questions, max_q=MAX_QUESTIONS)
lb = leaderboard(all_rows)
save_outputs(lb, all_rows)

show=["config","accuracy_pct","mean_time","std_time","mean_in","mean_out","mean_calls"]
print("\n=== leaderboard by accuracy (mean judge score) ===")
print(by_accuracy(lb)[show].to_string(index=False))
print("\n=== leaderboard by speed ===")
print(by_speed(lb)[show].to_string(index=False))
by_accuracy(lb)[show]